                # Tunable Transmon

                This notebook covers the flux-tunable `TunableTransmon` API and compares it with an equivalent SQUID built from the generic circuit pipeline.
                

                **Audience:** readers familiar with the fixed-frequency transmon notebook.

                **Prerequisites:** the meaning of `EJmax`, asymmetry `d`, and external flux in units of `Φ₀`.

                **Learning goals:** compute `EJ_eff(Φ)`, inspect flux dependence in the spectrum, compare symmetric and asymmetric devices, and validate against a `Circuit`-derived SQUID.
                

                ## Outline

                1. Construct a `TunableTransmon` and inspect `ej_effective`.
                2. Sweep flux through the symmetric and asymmetric SQUID cases.
                3. Compare the hardcoded model with a circuit-derived SQUID.
                

In [ ]:
                using ScQubitsMimic
                

In [ ]:
                tt = TunableTransmon(EJmax=15.0, EC=0.6, d=0.1, flux=0.0, ng=0.0, ncut=30, truncated_dim=6)
                flux_vals = [0.0, 0.1, 0.25, 0.5]

                (
                    parameters = (EJmax=tt.EJmax, EC=tt.EC, d=tt.d, ncut=tt.ncut),
                    EJ_eff_vs_flux = [(ϕ, round((tt.flux = ϕ; ej_effective(tt)), digits=6)) for ϕ in flux_vals],
                )
                

In [ ]:
                tt.flux = 0.0
                sd_flux = get_spectrum_vs_paramvals(tt, :flux, collect(range(0.0, 0.5; length=7)); evals_count=3)
                (
                    flux_points = sd_flux.param_vals,
                    ω01_vs_flux = round.(sd_flux.eigenvalues[:, 2] .- sd_flux.eigenvalues[:, 1], digits=6),
                    ω12_vs_flux = round.(sd_flux.eigenvalues[:, 3] .- sd_flux.eigenvalues[:, 2], digits=6),
                )
                

In [ ]:
                tt_sym = TunableTransmon(EJmax=15.0, EC=0.6, d=0.0, flux=0.5, ncut=30, truncated_dim=4)
                tt_asym = TunableTransmon(EJmax=15.0, EC=0.6, d=0.1, flux=0.5, ncut=30, truncated_dim=4)

                (
                    symmetric_EJ_eff = round(ej_effective(tt_sym), digits=8),
                    asymmetric_EJ_eff = round(ej_effective(tt_asym), digits=8),
                    symmetric_ω01 = round(diff(eigenvals(tt_sym; evals_count=2))[1], digits=6),
                    asymmetric_ω01 = round(diff(eigenvals(tt_asym; evals_count=2))[1], digits=6),
                )
                

In [ ]:
                desc = """
                branches:
                  - [JJ, 0, 1, EJ=7.5, EC=0.3]
                  - [JJ, 0, 1, EJ=7.5, EC=0.3]
                """

                circ = Circuit(desc; ncut=30)
                comparison = []
                for flux in (0.0, 0.25)
                    tt_cmp = TunableTransmon(EJmax=15.0, EC=0.6, d=0.0, flux=flux, ncut=30, truncated_dim=3)
                    set_external_flux!(circ, 1, 2π * flux)
                    push!(comparison, (
                        flux = flux,
                        tunable_ω01 = round(diff(eigenvals(tt_cmp; evals_count=2))[1], digits=6),
                        circuit_ω01 = round(diff(eigenvals(circ; evals_count=2))[1], digits=6),
                    ))
                end
                comparison
                

                ## Exercise

                Set `d=0.2` and evaluate the device at `flux=0.5`. How much residual `EJ_eff` remains compared with the symmetric device?
                

In [ ]:
                tt_exercise = TunableTransmon(EJmax=15.0, EC=0.6, d=0.2, flux=0.5, ncut=30, truncated_dim=3)
                round(ej_effective(tt_exercise), digits=6)
                

                ## Pitfalls And Extensions

                `TunableTransmon.flux` is expressed in units of `Φ₀`, while the circuit pipeline uses external flux variables in radians. That is why the parity comparison multiplies by `2π` before calling `set_external_flux!`.

                The hardcoded and circuit-derived models agree most closely when the circuit capacitance bookkeeping is chosen to reproduce the same effective charging energy.
                

                ## API Covered

                `TunableTransmon` and `ej_effective`.
                